# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Aprende de ti**: bajo cada modelo, **¿Salió bien?** con 👍/👎 para forma, textura y esqueleto, más un comentario. Si corriges el tipo (persona/animal/cabeza/objeto), rehace el esqueleto al momento y **recuerda la corrección**: los modelos parecidos salen bien solos. La memoria se guarda en Drive si está montado (celda 6).
* **Malla en cuadrados limpia** (como *Smart Mesh* de Tripo): retopología QuadriFlow (o rejilla de voxeles si falla), UV nuevas y **color + relieve horneados** desde la malla original para no perder detalle. Con esqueleto y animaciones (`modelo_quads.glb`).
* **Detecta qué es**: persona, animal de 4 patas, **cabeza/busto** (mandíbula, ojos y cuello: Hablar, Mirar, Asentir, Negar, Sorpresa) u **objeto** (Flotar, Girar, Rebotar, Bailar, Saludar).
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Textura rápida** (opción en Calidad, activa con Turbo): hornea a 1024 en lugar de 2048.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNrVfWt327aW6Hf/Cgy7ukq2MiPZSZp6qq6b1E5O7nUST5K25x4fLy5KhCTGFMnyIVvJyvyt+T6/bPYDIMGXLKeZWfdmtYlIAhvAxn5jA7As61VSJNmJWMA/du6Iw1/E2o8iXxyfCvtvZbwt/fj49PBIPBBH6w19L+RtUWY+/nwWyTiQmXtw8FueiECKTM7LLE/ykwMhvhfvkyDJRQT/r5NA4r9/ljLwYzH3s6WP32QsoK0XF78JH39s5EdhpzJbhJF49fTvL1+9EQ8AkhBPz9+/cUbCz0MRJ2Llb4XMoZyPEIpMijL2aQhii2DycFmG8EG61I1zaCiV0CuRS5FCF7H1WLX99ukrYc/9+Upi/1N/GcYAFX6eh3F564hIbmGMCfUCR+Bn81W44a6nfuZHMK4ToYBm0BE1VlH4WQBdkcsSqufck7NIFJk/8z/AZ7lOE6guNj6CehDIzYN8tRZ2EObzBF9Bzxw1AEYR9T4p5FKKeRJjU3M/DgCP/6onQti/QrkfqCqUyQAzMqau665STRNFYiljQOQ8TOJ6Nmq8ZTIvo4LmCppfljgoKogYZ7RxVwWMBiYmXyHuIj0bNdo+lHkhc8RUKoswwyLrMF+rgjIXYZwXfgz/ycQ9sCzrIEQEFdhhoKkM+uouSiA8KAoTNF/o78t59StKZvo39iQKq8cw0b8+5Emsfye5/pVJ/StflUUYVU/bqkhezgD7c5lXb4pVJv0gjJfVi3BdwfkYpkDF8kA/x+U63WLP4/RgkSVrcfHyXKhvL9f+Uo74nzdpfsAFXFFjYBEuDw7+dvb2TEyh127qFys3CLPYX0tbP/uzHP+1PQ8b9jzHOYCp9M7f/Pp/oFbVWfc8mV/bzsHvL8/+eAcfbAtaiwtrJKyZP7/GfyO5oOcsXK4Kyzk4OPhGvANCzkKgthPolo9Ek4tlCHwEsyoSsfCjHAeXiBRYHEiiKNcz4AcZAd1kmzBIsgManwt87b189fTFmXfx8u9n59iFxw+98XiM/x/g13cvT3GcR+OHT6DpQC5E7i+kR83a4XrpnBBZAY387kfQJWD6OMlAbIUffaJL7iDOGH61C3/tx8kIGRP+XmDRInFcpDGEEy6gOpBIzhQ4pzbUbLj0t2oQ/2R+CHwA7ZbyLMuSzLaAp5VMwNEDMowebKh/gEGsejMSKxgWAHfz8KPUbd+In8XxkUDuoV+72jqvIEuUUWuQh8CaINxArkKr9jqMw3WC4NJbRzVLTXwPwH8R/RPwBQ0uMxA8sm5g7d/aODwHGtEzWIOFEatxz5N0azvmB7dYAaHEfhjZtq45qmA4ehrOn77+9R9v3lUNYk2UszxzMRDx2xfPkGThn6eW0992vJFZYasiVCKTIFBi/KwIbR5JPy5TLymLtCxy+1rKdPpkDLQTxt4ik9JbzqaTcU2Bz0DI+iThDEkJaBKbUH5AoRmStqpQp3il8AXJ+YoIi2xbdzrIoc85sL4M7AAJVgQ4SJRvLv5VsfyHJIxtFg/um9/ee6cv3wIOvrccB5GkS4U5iAo7cGrE40iwCZJ1IEjya6/MkbuasByXCoLq//7743FV+2YF8gU7CUQgbNA8dpDjzCOykI6p0s8myowJofFB2wF0LkntsdP4onqUrUGpSzsYiXAJrC09ieSYT99npWxWMIYpb8McpiwA/We5IH6tVqv4B8pmcp1spFmsUerLUSNv5zItxBn9A+q0bj31QWkwgf0JWrfwomSZ2zURvZkj5Qh/E+ZsI6EkAZE3B3EkwESoSAtkWQKyxgX2Q4UBcBNkS4IThIsyh2aBZV4/f4NaeLU9DoB1HVL+BFYirCWgKUvEupQ5mk55AlYBfPA3KPtqqci6B3q6RA1nvLrxsxhe5SzU1IMLOqeQmX60LZ44YMk1aEzA39Ryvzc6736vpeKdAOY+mDxJtp0+J/X/B5fg2sgbqARZBqgBoxxQP9185ac9r2TrHdi0+MagGDVwdymLc/gpMxvbcdwcnuVGRrYu8MfTt69fvn7hdNmYtDhMC8yLzHIXCSpv4RUNggB+16wPD9iGB7JqluRhsWXaV1LzDhrr7wKYnHGOam9XLwqzF8Vf6gVTurcGRstC3yOjRGG22zlXFdNdwtK1QmL5zBDuaFqVBfI9OJhH0A9xFsPgJJegHnmgHgvPs3MZLUZoSy+mr5MYGAmGO02zMC4MAsBCbgqiAMuhWFMCIJAFMKahxaggQICiJg7pNQibGX7AZppfiAqHv3jrzY6Pxc6KA59THwY4/MW7DmMUzIr/q48fkhmqo0+WPy/Q8bFOBOhDS8bePIl8fPrcrOCBiwXa7wYqPQersG7rGzCQ0esBBSmDEHWhjT/AEUg2YaTEFAglsDgA2+B4FkpXNuFLEJXYHzLkoP0nLvZHeaXwfPyIXszYJYIXj8dup48AhAjTsKf7lanqsot+g6EqGoRsds0t0wAElo3lgSr8wE5S0JDNVh1DFw9TdIOtqzbY6QIEfG6Tmh9sOzinTwoVXpokEZSYL9z35A1cwPPZLbjsIJhtsOG8myS7BiExPQaDkSnj0lLvrKu6y4COtX8NU5flHYSRFvaSa6Wra+aTcQ7SmxBhcBmom3chqp95EoTLBJ1MoLMSbXnlqxrRg61fRQ5ALSTrNEJagC8VOL/IwhkMBzyRUqKXTjQVo7sNXi7pwEyCFJiHPrmguUSVWSk9bZzwbHlBOAdxUX9BkQhj8UEZ25biRNAiyFDOzlLIk3cUrHkQ9ZI1VAy5EQrs4sahqpovoTrRyFA5wCm1cD/+GmxV0zzAvA+nDcEj8sf+DbYIJJAmd2C7OAo7JcCUDFDd25bJLxaY0jmVbNmwl81iV3+FsQ5YPB7+hT/MGRW7wVyj8PFuJPrwuVJ5mQQET2H4Eky9uHhgBPgsmuPO6/UGpqLBrudg9hZVYK8Kh2HwjGJsxHGB/OBnURXge/v0VYPHSPOvSjI+Fv5ceqtypg2AfO4DMeMsA/9ldRXwOVAPXV4Ny2C0BjM0BZswbMelgXfdAZjyjL55IXlYA8Uq2HJDpVxkJBQcA2Xr/v4AHQYr2164FJMh2U+gFtzcht7nV4PaAE0z2aNpwNKwrdckL9MSHINIShJoVSTzb89PYEpb3pKykcYHNVsArWSBTR1rNhMDsseNNzch6EvSZlgc40MzaGJWLhYSbKfldOxgbxddnLDDiAqhH18zaIoVmP34ofj5Z3HUcguN+UKHfzaMdjXCuLdAjPOB/urMqTGAJhEG7lz8yzDraLT9HD15TCOVt81+FEnho37Ny7Utb0FFpnYGLizNsNO1F+2FdcEco6LHKiAOzHIiPjEw5V+euJPFZ/HiGX62ukNbWJ+MEYhDUZy448Xn3NAiCi8E1VDKrJ7SMJWGhBhh5GwkoMIscVrmetuJ0lxbiYzT8P3zKLl55RcgGOLlBYCOQsPgxLYAR3eVd7E11FjgvMBjYLfoWPVykUQggafUX/CyPAwUghTLu8ECIB0aUHPKvhHPwVVY/X4KKA8k2iCLcE5LEbwK8u+T8S2FczI/DdUqwKD40cNzZeyDbvAWCHoTrO313POjZTK11vNWtEGx+/ttyhG3LsDeVgZb6rLNHgKlK1hEhRa0nYIwT5M4hIZ6RMqe8O8NW7t+MMqaXsmeZkK9AfcA7CqaZfVEJpT5gibcfPEnGpbFlry+plb7FRnQNDZd8R5rC/uRCrP8UPXbwWAomKwYT6ZlD+gauIQGadH6B1hPIvcjWiCploHEr4AiXMzhEKoo/PUshAK0MKatWwBegVKrSRzght6MuTvaW1K9+JUCzUFDyepFCQy61NKOQjAY6AfbJXfnZeCDoCrWZeQCwSc3XrE4PgL+RKTtqBTH9y0+A9tiBY7ddbt0Q6ywv6x6/gwgLLMEEPeWYnaZyczmfPZbaI0S08rsgn57Sewt0xJsNgwfJYVBLXfrBWjbcO17W264/p1RNGHVTVM4teHkDwMnTiKaBaoS5jqtolomP5JavuO6bkvu9EqVVoCBnSBDQ/SbjYOqeOiPtVLVg8MgLA43R4fjQ0IACIBugPcL5Auj4C7B1UMiDbps01hzdr5obnhWMKJJ3NozL43Q0J4T0IfQro+nh7Pe0EiGyY7iR18wOgCsaG/jZyGtgOSFn38B8akI1l7DB//k/hTYS4TrzdelwhohuyixNS12g+RAOg1MjNOdtS+cs+Zs7aLNu6ftv3fS+qasS+d/YbYa8zRC16rMUcnSCsmEEhUGZo9Dq9WU8GPvfDS0Ha86dEzoC6zeMZypxyXFYu1LAxfUmClFr1pqRQJ+wG12wILpr2dddbzoHP1xWkHPd+Bt0UdU1Hthf/qOWv8OO1P9JIjcoe/Yovnucw+l7bR7Bz90At39CN3tWXSIrl/kNlyPfl/1btPeoCKLp05h6GRnl2iV/qtioHeQzsFu8Hq9APrb79SD23z9tdyRfZT57oUO03Sr+BNtvy6P1qsEuATGgU9wLcOP7CSjEQkvzYB4p4FLK1ks0Fuxrk7u6Kj24cjQ9+Zp6amqLWcOueYOu0cl5r182sSXsOcgwTJcyNrimnKQGDlajtUQZ4iRPku5G2hLac21If9HbXVlvnB6w2+pOQ0kQas3pjk64AK7RWJbgDLL6V8LaRh09OXGz9ZlavdGY16pZYYIVCGHYu4Mq+jauNhpG6sdqpnuYsdvsY8OnB9g+lscgo0OXh3nDp7QkkYYsRt6LbMYphxn6tffTp+KLRBAuJTG+nQuMKiQhcWaw6/z8vT1a86RBBhpBr3OdB6ej2mGPsZtMckySub7eYsqWNNwDBpGiUk6VFgpPe1YsStIXmaYe/7Gh+EBRdotYmBn/07Xqxvw7UukO838mybbrwEe5xHF8kYlAI3Al54cjQT85cDv8Uio/xynk6tSgXXxLztct4q4MorCNJe2Pfnx8Ug8BDDHx/DjaDx2KP4XTW34PRKTH/GvR/DX0aNH7YaAajHhYxkhnJ+g+uQnBDRBiD/WgB7DS8xImjzuBYMuJ2P9BSV2YrwykJtwLqcWzoPlgL8fg4EJwg1EfyvQSkFPrh7GC5mhXiDJZEP7/N4vi2Tu57gUg+BGIihAxU35IwquYvLY6Q+M2ZRLNg3XI0yH9OoW8kKm+RRmI5lj4o+XSbC5StQP08nRkwFbEUHMwZ65zqdaYvGjdTVSWa3QqekSoFI2l0f9xOWhtcxXbeO2JtR8G89XGYjOj9J2DgYMHwzhgKkMShNZeR9Rca/IvtDA15ikm4hkHRZhoM3PSsygbvKKRKVGVmkN65Qkrwym46b0QQ2RYOK2SuqjjG2ehNB3xfPz3/7uTg7z+QpETySAZo8eAvFxxGkEcnFOBSuIT1NabTjC9UCMZ1XOw78KTtSW4FOMtBVyGr53xTvJ0g7FHlrXUbKHFKIRoRwqMpsfHDfHZT7buTx5OB7X9uuijDACr0phihcgIkSeEvMV2PLzQmYom5LZB2C2EZefJcEWHB/Mhs4kBuwxHxYT+G5GPdF2K41AbeOaRiHFrAruQDvJApRWUQZhAqhargD88SnglxKz82KLPsUK3gPWC5CDTb27hHcFDI/n0Fyf7Pcjeq3gZsKRRubzqLzt9SYG/MM2FRRENDoRdMBcV5rCbKtrZs4iQNchGA9AF4egCPIHzba+zJVnavFMMTTTcmiXhbbJ/LW3nIG/9MtUPB7viLKTlVFWibx3W2Xt+Pxdtl01EzDvnJINeAQCQBh/PdwOOO73bQ3OHLCp2+4q9A+poOWyQv3dBKaHpbxPa9RTs98PA5FFrsmhqvCgrni4mQDpnGpyh195ARpSBmbyADopo4YdA30xGY+9mKlglJ/cqUvZ3GzoUDDibfzh7K1LuxNmLHfYKJRAoJYgSkk5oiqYjlHI9mlNkNArWuSfTlBc7+SgmzAoVlwOc7ah+5g4DqBA1yzhy9Gjxw3V6bg03fmlIWLNnppitleUsqyspaSl9JKBssbijkfJG/lSr0YmSTG1HsxBHqN7+h4/vnthtZQavxX270/fvRcPBD2Lpy+dE7UIA1qOMq4XIXAh6id4k8NLDuncrX96xLOZYjLsSCpUNRdH2oIbcxBgmDqTPd9ykkwPT291PnecYyL7mPHj7FOwkXmDtWAQ+RxGUeSW43RtajUNbqqYM69+6RnSqFLY7+V/AjXLQnCGjFUdeH776tmLJq+08/BUH2vtoTNZ+gLm1CUvU+stCn5H/TTgo2rHUocT9yFgwBDxrtz4kT3UCjmVjRHf3U5FttRM16BQdnTTgh4wDDWxk48q7DXu3kMCryi9on7LuZMW9zVJ9cg7AZVuv3bqmePT4ZwUTlushIHyNhvSgLblkNBjKVLLyMaGD+Zgueb9V7s5W71kl6C5JEmSz1Nbvqrs5Eymfqb2IjVTRXSL7mt/LYP3vL0v2z6HV3ZeLhbh7dRy03iJHpOMZCGnHLmmnJkWu9MeIX8DqmDlUtp5KyeU8lbVt2HJkoYRpS8bXVaZO7OlRyGhaZy6fpb5W/tygvpF/3UF3h6ykxfLYtpiMWMXSRj7UdRqtN5hQTlFB1/ol/bo1L31aVKpeuy0cjoBGS2vsE+ZYrO4mUlR2RF41zs1aktJE/Pa+iUpDv37F/GIA98/umMQBLmP2at9ilURo/ue/7UTKOOCvw3WrQ3TRU0cY8wC5y5H1RguywR9KZzH5HJyZSrV/hAmOkUf5Ug5J+10iE6YdEizGV0nJ2cqEH/qQceJwNLmN2bAtD+S01wXcdGTwdkpQjKiGYrMXESLetJ8+QpevaU3BwNRVpVuaqICyURvxBuuwQ3pCtnORio81P2xVc6pGdPgYiNhdmaKf/VIfZw8tJ3NCXP+crrgHNNOeHsm745E1aFSfNtBBh3ebAUzu4aMEpE5baDt+dKUvfhng5E5ruBuwgzzvnnbyNYeimpxxgqV8TAFBXcHgTulM+Uoz2so6MgMOB6K6eAu5pJCs582axeciqCRf/cA37bS8saclvef/9GXlsepeRgp+rRUEBRHfKd8z++usLp1r80AXyNXWO/gNdOFU1PLsgj3ZkvOomvIBtxSuUMoNK0ELN/aWJsXmdNRd1WElvJLcZNvawtnawvwQU/dN2mOmwAXHm14AsOhVXTlAzVE6cpXO0JpAym6fhQQpsWHSh9ixUvXdUfi+MpdgyHniJ9BHYzNgVVI4qpJUbfQN746K8hu70c1bfDdO1Z58jM5B4YC/gVaxFAWHXyAXleGPM1HA5R+kOFSD26iWPvZUtbiGhHQO9KqxBaU323OxeIk/iizxPZBh03GjagV5tLe5q3ZvAUb4nYyElv4dzsBGLc5Y3BEv0C7wq9t9W6r3vVgjJCQJaltI8wtwQVnc0Jw4V/TOOjZXE3srZRSvUP5ezBwJk+MJZw/+1cH0MgcXBnI/wSJD0YCU7Rt5+JQ3IAQAsGAVeFpxU9OJ6MRqoIqQE1ssyOOf3c2OpMSJoQfPWSxlf9pWMdpFBZevpKy0Pocf4Ofl6E+5x39I9zGP0Lfe6Q28jfc5tNwA/4w7VZfJR/Id+AcED7ZIpaYv56j7UiJyZjnuEjioOUvL7jlnQLB2KmULWd+kxWGOLe1LZja6GzUV6NGicJIMmQIfQNLa4jPGkyA/aq4gNBelZvTninf9eOtuVqSyyVQCPQEkIg98LOCdieohJSRoWZwtTQcCdo7IMHqRBMU9xNHTicSsKlECUPuWTNttcjZrWFrqVihisHtBtXZMmaO0PVTwGUABI7NQTtOg//7IPdVQzmB4zVq/0FIJTcPTNYGWhGPOaehUEQEXyGhQTlgLLSZYYb+AEYeu+OHV215hMVJWJ+QqARiEasS5GV+gimzeFoF2MtgrOed3Q5YmxjIJf6yrVE7NqJ7R+5JCD34A0yB2BmRiIEXKJOqt7zRI6SNHrjl2Y6durPouRGoRK9kUOkES7f7UNdKyqK580/vfwahw2LXQQAfAR41MKIet92jsrjEr1dVUiwqf+QAQ9gqGU5vUViRW9FJtlIMDhC/illCQYzadUHHNdQxNzT6OWiJG4byffcl2KdSOX3yOdRTKbg4EHwIyjk87Lno8Bw9L5kpEDUSsGe09GF+th0b33dK9XanU3jQtt5iObBmGx/XSEjVJxd9j3eyaGnUNTBlEHjkUjbLUtsuykhwxnPGtMtI7oDAb2G89JTpE5RApGiBe7q6vaMKzqYH1oQHDkeI6VOehDnbUWUegSbwVgkwKypw+oGu0tF4PN7RjpdkuGrKhOLNkxU5kVHbp0DfY43p6XS4EGPG6fVbWuEhYzrbzvqa8ChvYYjwiSydNeHSeKFCStMWL91zfbiaw4GQm+pZT8Ctd+lwx1D1cAHDYEUOzTRDZmzwVLuL8NbjLL+8Rd+DFaJIT3enfC9+jnYvfVHoi8+24hV0dm6PGkqskimkLFEJGBwgfqm/n/RRgCFIFBvXUgp0/bSWWG2pSYhubcVSwg7XnvMq1slJEe2Yp9qghccagcPbdNE4LIyujYqgsnfzZasYvfSiRm9mBqlYLfX9Uh0qdTUUsO0A/EbH1as9MwkYg4JZmabwgcQgm39Ce8Qx8u5jwlSVejq0pMZT0jVx7r2Sqvu3wLDDnaFtbe3XCEI9Zd9jZp12NJwA3JtAOh58XzBmXUYFLtF+uj4BoxGtimtlsFJLbljINXA8m6gGpdSWiM5iZzOK4CHzTGo7GiiQk5d3JAnCi6tqq57KXN6dYnjliH+2rWjdF5X53OoC7ck16NYEuaPxRsvFVTtHLpa3hZ3WmZHcVjvDsaYP3K1I6OK0uTtTKQ2RldP2iL56xa5qC6rZ0ihIQKgHwRenkDs+g4564rB5r9+In6fgEzFOHn1ZvmhSrGTW6nATp63Od8fSm0Gq4LaySBtvEVX9SRpUrCeL9K7UDyPEKNdpseXN5nY7isNsBZ1Us034awrI/5kFkr+QvKckfSN/b8fqiaKgQZu6m+GHtfitM9qZ1HfwRaswTnPZZ7hjvUmCPQs0rFB4Rr+K26Pi7o2sPn1UinZ8lMBH03LX0o1lqcUWPGgTMUy2T4jpGDGGeCKxSjLQnEHicmhpWh3nynsFO9GdvRaCepXsvReFoDm7PUBH/Mu0bbfgkog2W3oM5O6SVweoyabkhTd58gq3yvCr6yvWhnzA2MABmaQWSU/i8SHXzo4jKr4Rv/N+qzrvMhf2IqOjWH/AU239CGbBdR3BedxgbuNJEyJKhB+ADSvbvnzlxfsU56SJqPg011ER/E36mGUQPgJpf7H3oYkGg8uNDWR0cl3AtvaId07x4HockTs73+hhHWvgDXpfh/UStakf+e5abpvmVZIWZqABxaA6Vxa10sQQ9xWN0KmqTbyhkjepoxvz27kTZaVPlLp2gVxImjvDhTagsWbbAt2zy5MT8dOPBgZXjbOpgnKd5jYOcURnTXowfHVoQk9DCvsrdyVvgxC89gIbmBxd1UZqVsYKfRx+prQvfcAaMIz6STyjflOMln93473ezmB2e41q1LVrVCOsW9QDuWLqNxCx7gfmpE2Ojlvpc1qFTOnEI2h3aq1KXC0NAzyCSKfduU8wEwtEFGfpNScfiiZT61cfLCw/s0b8AtWfbpoTAXSXAMkFLlb2jEmJNHWeQSajUG6wxSemoU/hhWWG8YXIX88CX6AKOREKFbGHMdDQj3qQTj1TYQkqCvy81HjDnOVId4v9y6mRzokSlhWICmuctM9b49O/aoIyD8LsNShpD0+vBGqkz15UIuSgbdRW5giqnYtLi343zjEjylAl1QMX5QezLEcIuCj/5pL0u5HaQOv1WAx/caHWdrGKBlTB+pmLV89mJUUoelzqSY2MnxrdGLc20Zjp9EQgNhDrESixfyvDghK/aXnHMrsJQouP2iO9eGKEixWLVzzo6OOjzULErmaZrvHFqtSsRIzeBMwcbxbKOL++LvW5d0mqX6qa7i27GOZymlpSqiWQo73fXvfDlOz9JlIDnWhQgDo4aEOobI8BM6tzMvNzSmnVB61jBiCpWWP6ULTjQbvhXLHBtBEumOqoAYtE+ptlIvzfTtPtN5sr+ViHHFBIwv+VcOR/KvnI/4wG8IOikv42xWT1q5aV6t9+MLUInSVJZFePTpVvoz7wg1OLUvYb1JMzCB0krYY8S5yRlpv0jn87tbykt+rB0YKT/h6AXwtS3Uv1bCa1a5mHFotpqzQS2yvy5jPByCtunAhdfwSqvLTgFQiQIZvvbXUGsYz5lDG06K57czkbgGvB0z5c22mta+08HpBEWV5kCxJn1rf/99v1t4H37d++ffXtO8/Cze3X/QdUksHYOpWytjMo9YiOJW+0jkft47GKvJ3HXRXryHJoI5lRGdUjdltpSpTFtlK6vCYXnLSCPnjkX3ugF3wSoEfoHw2NAosMDoNWAfvidSf3tDU3nHXazGCmlhcW2ffep/gzJbE6hkI50+QojaNavxFvMfhOsikCX3MsvuVTkNCTwXM48X4MTERDE8XPpb7+Ay8UyMIkM04TQto4u1TnUaJvZp9dVudRcsSOeVnldyExnFXHM1KNhzWuPH28Cq4ew6ex+9MYr+aoG3iAbRoVWKPrilWNO3vBcBpH7RXh/NoGx2iGAnaOi205ZUQlaWuuwA1/uvHjjyTiZ5ikIPIS5mZNjmKAqT4k9eniE0Zftmx674S8ycBW2vpYPjK4oAOYK5fjsmGXLG5VhBAG/mgEPnpzp+MER0pJszSiSY97UpkcPs7eDGr5uE5+S+Pv2cCA3bnxwwJafOQ0MZiCBya7KFzEbckFMBrXXpxtAGktDFAedV2Ej/qzC8yWKqY4VyMBv/PpwJQBh/tggcQ959ID91Dug73H2puSmou4Vbg3v7rCT95Z4lXtEtcaWKtOcb6sjom9wiD4xNQW+qYQEPbza0yobGUeGjbjMQjGsxjzYvwTfXsP8Cw6OLhmk6gjy6xWYFK30KNg2p07NDvXKqXPvW0NYRC58UadyNgQfSQyDXm5Qcsq7/OsyYYa4iAdH6VwYKDi+USggCdgFSUzRoZ02TfbwOIoMNrl1Wlyx6cW8o/VPmLJIsdpU4d3rIHTO7p/CNx//gefVGS1Vj8AzL7dZb13Yp7/s2ut64v2eTaNztb6Z8/MVZK+JbDyYflUTZh1Hq7TkNCPmZUZz4S/LD/ILMmtvvVLkIN8aGR1MrWMeeOZEYq1MeKEadfKJlQncbx82gfxgs5woKztjJaY0b7IuRU68mECYjmKEtrIPffxOiy6dWrWsxr6jfj3R2NByX28Kp7h3hO8mSRPS8r0U+/rwy+3Pu5jBiJ0e1LHFolHY5u2V8z7nKS6+C/ieIx/vuBsn515PsYy/CCAPVfsuYM7D/XZ5/iabi4CNEk0QOdw+42MBGfnArvOhOK+Nk3KO1FHUyNvEUtNw44MY4ummubGXUYzy/lqJ4gyDdGk7x7pN+LIMUhcEyQSYq6oOsX7AjBQTgSqg894CRxYksBqiXMnS5PdNiahbNHsh3itGzLOp4o6T0afodlPnOwBv6lxq5eiu2kiPSkiX4Vy96TaAeYbnH026y0/m4MEGZz7QQepr3rLP2r4STYTmsVlzMkHOZj6YnLSkDu4TCrvr4trI/xkyLvGjwH04lqZ9ayuG0QyGbHRT2q7svDvqbOsizAutPo2l0tePr3vMQcN3aoXB9vLgn9Fpeo/tCLIOFRBB7YDSOLc5+yDHjwPpeGwpuYyd+ppDXcXRbP1qRYAw4+41PmFxD0EaTeda4JpUrqm86MTIgLafbm/DB/cQLV3PsJ+G4waG432c0X6LPOG/W441uhzP2r62j+oV8pYO+hyMgeuiIyazTfcbwN0/bIGDu8OjNCEvi5zUcqMl8XVTZpoYJFDAzKIswaal2XSiWmuAemUzm/HhM4kr+4wPQG3HIw8olG/uiJhhGYfOUi8Co23FXK8sAZYzIZOHY7TduDoDio1OnlRn1FdXfelzpWm1ACeuRPKleceiTjU6/d4ANl2bubLE6HTsrnFY8SlcKrHcRAFghOodJGrwUuCzMBpq5A2klUhHUetVyhyDCfzEgXefUIXk4B/Dr8t7FQAggt/c+eqIxao//WKgexA+dyMOUg0JmwVrWgSIdcFeiYaxCiI8WYkbp2DAaMEJR44X2P3px9B1tEzhkQatSk40ujJZrgnm05PNjt6or8hsqh8vjQZ5JxjdMAFFHKUqJBBToLVShd+niAFqeAoX/gp/FkWZjVhGbBkrC9z0lGCyNfMgfaw+3XCsdWtGnbDKtnBIYD/ogRPrXskQzvayRHWan2/HWrK5v28ubjrrkKo2RNuM9EAJdrjrE6yMe+fcfNyBla23Q2bqFK4RyUdcfqEsUCjV2bUMoax8FItt7RWofV6SrVughOjxQBxSrOCtl5spR8rm11pSHNVmmV99YguirlnaNG5Ew2PTlXMXTF2O+EGrzFWIun1G1om8+lSGro5eol2FYh3GRvH9OOxNTE6/i1b5RvhL7EKXcr4j5cXRNdBRddx009ebPaeqP+uyRqcMMbZiOVJ26Xky34JD6zZUAdolVnruDfP/vdIvHt/jscryY94djRfALQV529O6zm7/X+RXAsy0Di1oTn4Gd/kt5Au37q56/hJKvvDVFj/jP8ZY2hucdutda/MKo1l6mHLXy68WdQ1kmemoVUveRj2kLkOou2hWTQ8qio7yD4biZ7L6wCBN5Zz/z3rOm7U3MEWduwajkvApGN+adK+eK8rRMP22olqpHUDHxQzoKS3u5v1aAb2afy23fptX9u3Lf+DhmbeSKatRL7aTGdqhbd8I1nzq9NNZ/BY80VJ4HHUAg9PIrbEX3+WfpCrD/1pDAwgvB3wP6C3l9ceklV4iz/uGAtdZAsNX145gKgCt4J24Poe9dxXDatxGjWxW/BVGW3USKeMM3D2DCK8QXQaoYQDCvZM8eyIKLcdhloFgdCwoLxtMgGmKrCvkwd8L/Sn7I2MqltdPPJmpnTCmV3oEPJkIHhe1VIQq3oEE2rVJRTrViVm0T5g8YgKXaUpLMYMv5vE0AvR/1CC6ZdPOUcwldkijKYXlxb/olNW0xJfUMZ/je9agCBSlQwJU5YbKNhx+RJPe+WkMM/P52Go884UEU1xT7eRyoJbc7vcqvCDshMhI3z3JgthnlE0O30pTz89weWrdeqDFcLxGVDkBot/JHlUFtX10C2XBPfWUuMf0x74dICSdY6HczUu36oScYIwm3L1MJ1+TEdEnFPG07XcTq+1SlvQZcfTxUa/IIN9ynZ7yysA7zzMVx4bOTbUdPbYwbVRiq4sujqrFpObO8QktdkWkwbj3SGHdwnWjTMgjDZ9QDeOMyS69pGzm7vl7CDEfaRdv9Tb7CH17hJ6RrcurUoIMC2QIaBb0d8G1gZN4bnpFmkw9dfgaYM5Lkn6cmfjRfeWnurewT1EAMjvvhPVKmW5UgKistk2DUmxB7N/0eZBno+eFZDByJr20CmGcdU+zL7lK2ZrXFxVbiffnM43dufmPbyK0xY7soAWG9omjo2CvRBFmMJpN+SLYfh14r/79qaBH50oRqdr5UNH67eSJbAkvftfiKZwvpbFKgnq3bTmzBlnVq1RuoOIR+EOD1YzRQEq0Ql+/whTOrYPShCBj+ovLy+807Pn50/fn53yFcQo9iO8EH46IcL72M01pagEptjz4ZB4dCOm35Fz2513BIqdNFp89/7N27NTmjoXr1TDztq25a7Th2jdqVMFLR3vYB8KTPTWzkhz3npH1Cn9UTFLl+8WeB59jQHeNoVPjfS1TIInPVeIRB6qNwtIUCFqu0CgXLzhzPXmDqffSxltJHiiaYKXOlcBhQc+XqiAISVaPixjHeSsgrnqOCTjer0CT7bEUBferhBSZLTMSz+Dn3bON8LKIszUGr0rfovBwz0eHz4ei7y5V6oboA32ic6GvdUGvZ9B16bfMeK5Ntwtn2WsNqXZwmtdp4wpT7hRlzxwPTk+12A33MJTIp841Vt2yuHt2B3Xb3WQodr90GNnmkUp430kfjwyYOjMdrzvwKnO1tHNskdP7T5xes09XMuzLpKYV3srUqHj7nDqP8HgPlv7RteaEYrvEVFm9KsvnrA7/jEEcDg+gRKxGZ9AYbYrABFTlP3LSabHVw7+Bz1l6P7/P44ydHbYTx5wT81uNT1K7FHbW6SAduWeWvXN7OCY9i8sG1xu1e4v1VT33WmPDn6hK9dNG1aDhtnuSbCrh6yiBAeDU2eKm0ukIDJs4F+zyqU1T7JMLsMgAV7JPCWPO0VrizTGmxr+ukU6YGcG+1uZTetyOkWL5rAWOvCC2RSZ2OlxLKGpoHI87+Fe8p3X8kbvifxbWNCdQBG2ZEZvD38R/obOowXZ6ILFQoljga8jvVT/1cvzMzA5Xp+9w1BdrXmVnDsRl3U2B6U56rDliTpsWKc9cKH+6EJzIcDFq3hqMEQoBIXWmXS5IUjQ+dttdQM9gXtiggNDdl5GBI9LDAKiAHamD71DSMfN8VEBviBqEMraTzk9r9ZPLt4JVIPRHwZRw6tjgc6l9ciyCQhTj7BDZ1wAR/Ti/JnQn/eGdyNnylp08RohC4FQrl/C9wrD9/2BAVO6i9ktAWt17vmzv+8Bp5K/7o9IBhf0SCcOrkpA+D4Q9MT/iBP/CjMtKnLYexygExDEk8cA4nfedi1zlQKZC/v8zamzNzBlg6sjta2nrJYMVrQcI6NImUo9nPU7fmjwlcznMh7gql4KPjLg1LIoYiHSD0Ut/Vf2NDECjuOtzlzAnlCn6XaFqtxueMsQ6Ov48ZgmqgOu+joAhHusMfo7r+hpVlTI/Fx7GEpk61Ntv3Q1yVhB6gld6D24j0e8+DjFQ1AsXtLU3ky19ZZ2tLZPbJiv8YaxS7Xd6dn52evTs7cwvkNy5Q4PF/68SLLtIe0mKFN6SQkXY/p50dX5fSvn2vZEVk23pMIOD7mBEGeX0AMPgCFelu1AhY+orxX+4KnyBPBWJn5gsJUzgB/4wekDV61BMu71G+UFYOXqmQHXvgB+VE+9oKs5wyFOdFo7v1Kp7Yy+w1pEU2+H9v5BSZV8UkNsn0pRF6AgR91ODzA6wBoNNuGpn7S0Zve2TJRlKQqjXqvkmbovvAHtX6aVkq5O2qgSbXR3avOMzg2kQrXGJytaNdT6VKfyGN4I3g1C554axgpfCqqOuXMvyKICMscBBEBaU/Pjy4szei+zzHz/7v3pm9/e35XTiJav2pE+Kxd0NOCk7hvGsHAdtGFoY4fRlk0zl3vTsXWtp3R2qVhk/lpSWgHVUXV5Vw9HYKznuEbb4yasKZQFpi96I3bWgCjsfwY/ONaIoPUGwtdVFoSSIHyxMO82nFa6oj+IW9eycccWbdhyj8GmpPM1XLyFJ7UnDm7Zeki3E/aI9J4AMV0zELfij3wxjDrqlZCTqbNMu3kvSvrxJdfD9oiJ7mFDQ5fqQb1uyG7mGNfZSlv/joSlNqSBi2Q7xHCJ4tFpz13PJW18GrB9LbcjvQOueTLwOh9a+ViIEC8rC/QpVwBD42J4E4LiAzqddvj0I4Nw2tvygFdoM14jCFHc4rZM8HMsVjREDA3/H6qxZ4JniaBsGg9kPTC0y0PceHHSPfGFPx8cHHhnr1+8fH2mJ+TgANU8yCtPxku8QAwKTtMMCF2hbxklMz8SqtqB6paG0tlnX4M/q+FViwOqMxrYfwESbGFy", "blender_rig.py": "...", "app.py": "eNrVfduO40iW2Ht+RZiFWVHZSqaUl6pKVakG2XU3urrTVdUzC6sTnBAZklhJkSpe8lJCAvuyCyzgnYH9ssDCxu4Aho3BoIFd7IONnQcDVX/SP+D9BJ9zIoIMUpRSWd1twz2YLIoMnog4ce4R59CyrJdRJpIx/8AuxIjZzxPuB3Gb+SJkIs1y+NGHH1GWxHQvijMxiuMzFjMvnsVszidBxLfm+SgMPA73Ivx/lvBURJxxBlfnIoVXWR6xLI8ABFw/DuPcH4c8EY5lWVvBbB4nGZuE8Uhfv0vjSF/Hqb5KeOTHs+KX0Fep8BKRFc3SfDRPYk+kxZ1smgiYWDQpbgQzsVV0TJNmPIWrra1xEs+Yw9QzmM04mKibIoLZCv1oIjJX3tnaOv765as3bMCG1mM+A4wkVodZj+MkEXT1hoe5z/VlJq++5EEor16LeZzGePWKR3GKFy/4CB+ebp28fvrm6VuEvdhi8N8dWIPZKBF9Zs95GqcdFntZIkSHeRzQ3mGZuMzyhLcd1usyagKLNRMAF2DMYljFmL3Nk1FM4Cy6tADaYYft3z/osLtd+K/D9roH99sd2eaJyHgYcp/a3e+ww95eh/UOlxu+5vNAttoDcHuHdzvsYLnVMWAA2xx0VZf3l9u84pfBjFoddmsdHnSP7kKz662nf3786uSrp2/cJy9fw9ysXSQ9oNXdF3l0lfNo/8nO3i5PUyCNXXHJZ/NQuAB0IlJra2vLF2PmjoNQpLbf7lOvceKLhFZR4snlEbT3Y2cSjnBR1N33OffT2j3gntqdJJg449FleccZhSLyjRZypYIPRQ80iOI/3Qz4kGeBFzvx6J3xdnF7loXGbSDORKRBHO37TopPakDngfjAi9HPBdCH/sETL4Bl1j9hAm4QjWMHmREoEV8fIxEN4ZU5z6bOuziIbL/Dxm02jhM2ZkGkcBiMmW4kLoM0S+3ld9oFyC8A5rgEkQJ3Cd9GeeDgn6V3rW2r3W6bE4P+xiic8HUACHKCfo2BZf30Isimtm05YTyxOrpbEBh5go01KaT5bMaTKxunrOiBw2QJA8DqtsXf5WkmkDsX17LzEHifEFIMxR4D5QIOUcrB35fHfba9vUAYwxbxp4vs3zrtd64lv25vsx/+6j/BaPzcC3wUmRZOpuy1fMtqMxGmgllw8UXR49hSHVBz2Umrw1q/bLXNTj7+Dy0Y2ILLpvC71b5m80t8aBkARfo+F6HIYmYABmrA1rZxa5oj9UBnXXggf3QYURQDAVkBabw1Vy+1dgBeG/s2HqZikoOMT90sBpGj5nHNUoOMx9azOJnxPlu0ZnmYBedAXrxVQZr5QCGt1QP6yuLWdWVYdlPXYwRPvdLo3iq04VpWptRKg1q3CsNuUHYbYZeN3ajGZUcm8NdAtyLpVxaAbmnEVRDyw+/+ib0NBKiltM9o+Gz1xEqcVibDbNIE7fqU8GYxG5iLSUhrZlX0Up/ZlyHNo/HlUVjMsfnlhd06D3wBBNZiX7A0S+wGKLJFqw1cAs1SZgsQKfIhm4egYAnbMKfKRJdeL6ZsVXiNCHMTkt3eVmsk5U2FqUmveCCkQZwoWVPIE4fP54AG2zouGwHlwWRQKEsZSAKlAuVUyUOvIrK8kKfBGCwzbGW19UC8ssssmJNhsLCm+QxwA5MHpQt6IQGhz7FHD7Rdks8F6XXZJxlxB2BcZDytKxhSMh4fgZbB9vKK7bIRCE8ycUCLgWjBZ+rqugAAompCoxmekTY4Q3Fuo8pKIk5m0SjhH6SB5IGICgmgHAYtp0fTPlMifgmlY7RkhJeBwpXmK4hOQgC95imKhxutdodVfravYTmXJ4r/fYFCn9nnaCkvcPnlEsm5gIQjea5mpuV3e5kkyKhwSVyvJAmtXICgaVVgHim0mM0DbugZA1JV0UjrPATEXoHCCQNxLtiKSVnTOIkEwZeaLAb5D3ZtyOzf1K2g35DQz/inf+Js+zGOK06321bDHMPYv2GG1lffPIE7E5HES0RfGSnIgjMUrOfFBAHRRDMddo5kI1mk7PHUCTIxS+12E/JnHAxlV+Fk9eBeywZ9hu0RMRFKVLAgmYf+BbpIqUSXEoNNSDgXCZhtAkVlyNcs9bNP36foUwFAbJkqZTBsVQDIFT7/9Ad5j1p/+pcIXtz1YJC7wNMTcGxWLXMYjERSLjMobslrQCGS1YDJQdKDPQBzBSlKhJyLqGli0gK4aVovqJWeVTEp4+XWKSws4tCYB3Y94iC8PSG9TAAK1ipvJDOyMHA1QZ3g3NaM5qSwVjyOjDFHG5iHxbDqoCS653mU4RyC8afvPfQgVuLXzgJyyiY8QRJlWQw2Jgwf/GAEjWhI85SdCy9Ax2/lZAJybFDc9drsEettPCGweeX6CupUwalOT90kvIPjfHnFZp9+DzQtEnzVQHFJu2kQufgyoAYJBJGsmW790PrglXbZL5RUmZE4SwMgKcYx4qANzxX41BabwQTLI8F56MesuKungTLiQo9VaWOwFEmpDE9XU8oPf/cP//t//o4tLq4VIOVAWN9F30VKQtErbXAoXr98Th67dZxLVcdzqfxekJLFO6W6xUhBEnCSF/2Kvi1Vq/W4qkcLzYpvf1PXqBhmgBkW6MRHURCBnYLaVvo754G4EIk7zWah7Q++BiOiw6YimEyzwV30svG5un0RJGLwjIPu6jCRATeq+7N0QlcKaZZlPZUhI7b/hNkYcxHgO7aR9HLA9zjhMyB3kYCFZYhKEM9zgErhJfBcwXJHfxjjQorq0JHzQQHRxQ1epaWCVg7OC/RsuZxquWAlH/rBOdiOV6EYtOSM+wv57/X88oEfpGAkXvXHobh8wEEVRTukOPqewEjZg2XKtNAvDMZXOyr8oFsihB0fcOdlYH31QfPms+jBhM/7vS50NOLe2SSJwXbs3+nt9fZ795tgj8if3sEIVZ72ewfwIqnw/p0jfuRz/mAMne6M+SwIr/rpFTios508aD1qAGXOm95Kgw+if3B3ftl69K9//9//5uEuNHiErR6h4rVheRHr1pt8hOE7Tg4UKIZ5HqacNXUw0uDVEMV9cSRE69FzAdKaJ0gWKLtfHgP4p5o4H+6OHqFHC+/LAdBfza0Yd8uTEPQULHSSFsG393mcCWryHhhtbGUDEAyZjXE9B//YbbaNogZc04KQkKRLgniPYYex9WfngwU+KJshudeaWX9GPNAr2hAbLDWSzIHIo+HZ9FtOJE08GueuDDa6fB7sYvBpsKiTcMskYfAkfrl4r8ZmkLBmpsQbtBbw97qlUX8R+NkUKKz7iwfLxC2pqd+tUVoTlS2TZwtEdBhfAPHkYZiCxSOiB+wy2Ulh+KAzd7IEXgEp06oBl6+VL8EKy+HXiLTKmWhB7RD/9RMc/gNQnGCB7mTxvH+AJPuQM5Aw42L+GTwX2aAFXiSPzlo1ArWqlHl/dH80Hj8o2aC3jzCPR0mQoLya84hCnugmzEMUTz/81d8+3OUldW5t3WE7P+I/NlFMgbqaCCUFiK+CFNwSctSB19DsgjGADmIxeHGBdLyK6Cb4+nkk/T/QoThOVPMe99FWIXC2VLA7jwqfHS4LtdB22GNoSLzti3O4eY7mSCmFgdnhbRXjvwO2BJuBmQbrjMoaRBxFugEelxNwGGodeBylswBDkIjJ8+A8ZlEAzZFxKdgJ8J2tf/vNl6QitT5yL3iQ2WnWYRGQBrgUCVguIk1N5QJeKUbIEC2gLKCjEX8XM7DNyH7AsdI42G8QxG/YLMaxotsPrfk5Go+FXrmYAvNRVzpsmGZDC3+Dq1DED/GeD/i1Tkte1wOzYfZ21zk66lCzMbjgHdRq3gB8E7jTmqEhguEtUyjtYONW1gVD0umOMQrTNnxxaJeGQswB7mFhdSF0MMXMMSQ8AFE4SZynSRIn6Cwgj4HVisChLfRMJCoDm/PYRjIuMZlO4wtXz8ONMeIKa5sGuOXD0XaixQNSk9tBuPDcn4EaLNWylMJBBGviZTJ4foYx+DpoZRZSMwcMvQjtHGHD0L/MsyyOHC8MvLM2CnfAPdBUaoq6xTK8PhviZAC1GGM+ky51SUW4WtKoUdO9w75V80Mrj3QQkSSRy7n40Efi8ESYhzxRPg7Ql49WacRmYgY0y80hmXYTGkDWUyI5xQ0+MMHbGCkuzT79Hun/4x/3kQi0ZaR2FD7+qQ9YTdkIVGq02n9QsutYBuaBHU/i8NP3kxg3cxQrRoUdXa64Mo4oyi6xMIe1WY7Am/OaN8Ts523iBH0XxCvKSrz9iB2CGU+4RwtQdQzWx9ylsdiguCPgZdQiHRaKMVyTGAdmmQqh/3FJ78AjwPO5cEeTDsEoN7XqaBlzMOppjwtaCgGzQAZ35eUkD3xkcupKW7PwI854JtCi9UW8BJHugvMAUFPaIkLwgJCMj0Khd9MQpPT9gQSLa7CA+ZlQAcMlwD5tmonB2yTHlrDQycAqZDezk0+/x82ytkWybjbPBpZF/bkJ7qJxZXHXwWpeGAALnWhR1DakJJFjD8iLR+AwGHLyqhTyheslI0noEdKSC9PuLjBbyh38BZQk92Ed/AeNLty6297e74Fw60maEtEEmpUbpHYhzIguWJAS2RB1ESHoO2VXyvC3JW7IELXaDlB8MLcNs75BGtYMVuRFLwnULZ9+zL3g0z9HjiF7C6nedbo9JcfBLROkPgBRtG8YKa8FkQp0Ef/wF//NFN/JVXVccq4DxIaD7d0slvuPalKShEsA4tITMNWn9A9KYA76fO1Ux9bXJLHmOUY0YbTJyrGCchDXVvM6lH2YQwY/KMhcWh57mWnb0nemFxTYdwESh9qMd7L4TETuVFzad5UJnJFLPAYRDkjGHUv0S0+khx755LbCEsBN6V6S0it+kBrusyWmUDob1MIptgqMV7IuXBvKVwaa0fAYwlBPcaxZEROIQx9VVQhcYGOT9rC/s9c1dC7edObx3IaWHeoC5K1k9DGRD8g84HODNpVZgNvu/NJWv0DuhjHPoLGxg4nPZrJlCTOOXFDHQC3SIPJrkHGq+IZfuassGB23wJ/GMC/i5MzkniWSRRAJmUCSBpI8+mxRvk6lqc5MUa9lO7lx4rKtBXwh1W+Eh1EXokoS//CnvUoJGFL/5lGu0Qo3vqw0xUCuufpFWxWlPpFqFFniRnC0GzdADBHu2uzhAJzc0lge4MXNYKTGGoziGGyYUoUBujTNDfTFzcC0niNo6kf75teUQkR5HqcTSxq6iU23Uc7zJJMb99ZbbFJsfk+lBrU2QFapSmls5e/q2YEKNxUMoG7dRjjrs0XgBwvklqVOyIqnMMAiu5oLW7Qd10UWdd1rKZ+XXiFg9FIB1pHuIVh5nl2dyDyRpFG8Vj4e4+ZN2MDt0r0B+GinSElRnJZy3tKVrVxrlB4ghzhweERmjVonNYrSg7OkPWo4cVICKfyWiC0iGlLEgFTuVAOTKg5pTQOiEMOylX3QwQg6sKLpwwv5laVilmR6racTFbJRVtMu2erFJhsCR284DnMyGKwaXYMezuc+yBU7BYfay4Q/UFOvNcSN44YuUNnw7W1mr3cRwU8/gYXVpohy5dH4qPZCi48WRdu0x1Vzm5C7mfW41y9NRbAf1KExDN0pYIWZSDqdNCPKXejC8FqXrYu6ofZWWabgdolInhv04wdM9cx1iETbaQaB6f21zUmskawAjNWp0sCeXKC3egPv5hXPmpuWa14FeYtVr4WwTuIoEBTOKPZJrphxCECao8bSE9PfZuH3+yXsjgkaO0q8KTjO6f/ztXdxXq67tPaJKb4lNXRolwc9FkUSHbzCe6rFOZmsEayMba871aaOpEETXD0AZfs1JaLt6orqkmaGJd1oaxIk8f7drlUcVKPbFW+9rS1LBPcBvXYfI+POh2AukU4HFuG2PfwwP21w2T/MlSAcnmJMvTjguJoXoMsmqlZR6DpVmwGWTvXXVs3E6FTEtUSFQoEzmx8AZDmbOtv88J//km1vv5JCB9GNFHJDFG17m7bhcMaVc3xV2KC3AjAviuHjEBOB2HBnvnqBuXjiw+VelgPv0L2fINjL5/L4Fu1y5WnOkyDeevvy5Js37rcvl4/fnKw6fnN8w/Eb4+TN4zUnb77RJ2++fenSKHAI5312VjlCocenT04UQbYlDEk2R3lgrzx8hC784lq5jQjB6hh7sGCt62MxikqL3vE58r1GShHmqi1dOYjlAcqoxCWGrgfMYrZcD7kPCVZ1CsotSYQnhZ08srPZTMod2gKkC2tYHNWsbdzcuXOHffxfb3gI5gQbgTT/5XfR00LDwhs+HVCCh8X5pGv4QyO/dtivwIthHsbswWfORL+uIVBF/+vf/8ff0vmCJL4CUDJ6G+GLVwwQRhbNXGS4zxEygcLYKVGKc3bjcXEiuuIgqmngcWAH/BkQh+BjLm/BVs8N6/M2dft5CWwRwwUMnuPuGWktPFyXcXBMgCgBJQJ+AOuo0enN4dVa5iQBgZHESpswwMQooVwEdSTb3GYszvybUV917egRGRhaPTwVrZIH8uIoFe9QB1V3tOOzctRlm0V1PcnSl1GSHGx/MsFiJOLt7VTMMCaMdsTxOYb6gJZ/+It/Yceh4BkOGcZHthv8gObkRL15Ds1XHq6w0nwUUABpe1ukcx76SDJzkZCUhjcpmv0Kj4bQ0dr045+cuvjRtlA5ZAyjf/yjtn1k0JN//FOHSYOyCNPhjxkgAQaQf4B+xzGZu0EaL3VSOdfwJqAQF8xOxdxBPnaInT99P0Gm4kkSjGhvCvCOa0bDep8H0qBeE3m3Pv6x2KVWbi6F7mHoiCqh7HEaPu4ZxkB6b3ePl5GCwg6G+hQPthWjlRLH2FHf3qaEkQlypu52e9uEdk0ip6Q6q3oUBeTLMTK6j8E+3eqaLVogEv6mZe6X4J3ftkCgoMZU1NcpaB0TC1IUq4UjMQ4uXfJYgQNItubBBiZloelorsbc+yjplEqE6eOxMjEFh5Ukkp760hGQn5HLYXDAfFoZSo2jp2loZmXgSpPSkAR6mBqBmeDeVCIMG8lYVDl8Om9EYYBjrTP6aqh03k54AZ7YSgVwS6TPqMJYQBNonwBXcRmgQlhdZskxaJ1Lc72dom4K54wtTcB9VgxQYSgSF7Xwv5MI6BcHQ7G5gnRCPhuBmBkD8fXLGPxYBeBn7dWG8E9gy97Whq6+LZEOYyplBFCx8Kaxs2SLAkLa6/hrxDNvaktrWGJIxSnX8phiDHptNXPQbsg5sCFPSVTpHecVezSAYBhDGONDDKkPT4sQOTADOUMiyoHdEMXUdbthbzwAuzME40A2KPbFn8V09hBQ07veXZQNrs2dGFgJZA46BgnaA7AVeQJpIs0StTBjB6Nmq8PYiRG+RniE1EFTfJjwPKC/VX8O8aCj6OAzIt2dVlsAjozjgeC0FEd5RpT+CJ4K/mpfU4bP8sNk2AKordPK7DcIM1b7/S//YWW/5paPYhe5upUzi/FEU+HsyiUplNqlEH/OQ+BcOoWsJdQEZLpPZ0ptsgPQDDoPUOySJoSW4wRPwLVLCY7+A9FTQUv+zYleMvXS+ebbt5jgp7K+cIMBDyjI4F57ab9QQwhS6aRXUYdxhiDKS8oJZpON/H905WGqrpyqM48mFvm0eI+Cl3Sr3WhLbJQLV3H89SHg2aQWYEY86rW34XGHLS293x72e4enIHDnIahU28JAicUKO1wRAoEa9u92T9Xioy0vl98W51kfBcgbEqBPwM1UaASU0jL6P+USbq3AlV5BCpLw6Mq+GYnFet24N1CogM3WVSWUYKwIcTAEDDkBGC+Xp1sNtHfrs6k6b3maZ0FYbufRT8eL51dVWPoHDIWW3CUN5brtpT5w19C0WRbXy16devRTe3Vznqb/f4bYft4o2WaRsR9np9SDYPXAVwX6LYJgKCemGJADyIp8P6BAuK0QIFw3CwKtp0A2HRayCbdg8sQThWKqErDknnla4R69GRcn3rS4eT6D0cqGznmQ4OxcssqujJ001Aj0muPlPnfguYtWEmLBpq1b42GQuvwc8/hHIcYlaTXwPEzX0OhNVlbVa1soFPnqpIotnHn7GpX06+NXoMXPZ06eCh9sKjxn0+07vfE1/Fh2XQEUtJUpikVjDJA+/xJzpWYOePN4Ov36F5QD+fzkW3DRuM/7zbDsybB3ynbYZNgFCqv1vqCH9V4qCZslqOJIKSWigKcxFTAvNACFQ78KI6jRALpcihbhXrG4RKJ0X7x885ZCqJk+BTKZ5/ryPOEzfW1cemULPJhK10V4U+74a1pbJi8kkHMkEhQec3R1uk53mS5RqpalKMgWtSJg9oDvpLOA7eyA25Bc7cBgBwg6+MBxvg787kiapEXX17SonfXnBHEqaJjniSAw8xgEmOMn/AJ6k7vGAy8970TxVHA8rRHFOQiGdP1mejoVYaiOsXl8jtDdOM/meaZuYtBHX4IjDs8G+7g17MNV6aOi4SCPQVxKqX9J5QL0kS553MgG09RqA7npnx3LSOZEvwj6mHck1jm061A+0TZcEzHiQRs+3MPbIF74cP8U/x6UIJAe3Owyk7vw6q1ed++AyHp3gS8XdyRNWzdoOgOkhdEu4Cr5yo3S5pyHKZkUXmYjGeD8ANjgHI81wj8FvwLi4bECBZeuuo/6BBd9kM3L7KYztyNxi+B17L4cLvHL8MwtzghdliIJdyeK58P+zlG3somKADvFfOFqhiuhvYeKjikdiIRjaKE4Cm4fdWlv83X8Lqazh37MeDjBUjKpTAIKAzxqDOwz5VeYr1ruNlfYq3kwGJjR/Lu1gUOly0BUcg8qGTatR6/kzHCrErOD4BdIe3KuVDbAOn/6XTxK5X2eocyKYJEtvIl1JCzQsLinikfi0HiLXJwv/lIVJr56+cokkKN7ikKO9iSJHHUlaRwdKkK4v68O24HXJmtT2BYSpJKK8M+dw9HhaNzFy1+ghWhbvwItY2lZiS2ORvdG47FusVU3mVVz3bo3PvLvjgx4j2V/nupPdHl3v7sS2luUv1oOY/uD8WjfP8LLj//4WLP/rzvsBcyGUtOOpLBFCwMowJuCT1sNVqDPmlX9TXmGDlkDMNxhOVmchKOSFjDsUFB/KcbPqSbMlCzJ6XAHJMajAS5MpRVm98BUDw/uH/i0fUSvkT0AHZpeHT6ounU03OqJPaM9KslpLemTpp+ltJel/HiQZr8GOYiBl/tHJMw6ixegu1/ATcxUuCRJ2SbZJnX4Wl2i4j2X1XgPTP+o2z+t2eW0EGVI4uE8Dq8wFZLNYWBZOmgtYKzXLQxUhYNWBMYwZisl8Rmw2gJTd9aMZGzJljuU1wTt95zDKnp7zt3r1u4jw/kmijDGk855pFl7zn082dTfn1+yoyIjqsh82qtmPq1MQ1201Frv77eqw2nd2eN7/v7d1rVK1IMZ6ryrHnSaxiE4RKsBe6vAtR4tkDau2cPRowXRISmoRX6NGXQPd3GWj4pNpxD1LYoZGcbVkqWwTLFYAjZ6xLpGHNkkRDJLQ35NBlsc6lz2nxW5y4ilQa5ArG661FDlHV43pUFCH+roSUqYLHDUUrJY1rNBnKIxS2CxHeFiGdOgrFyidbQnHtLVZW/Q6rbY5R5Q66+Buq/g9wLZsOv08Ode5admBDWXw0N9a8fn6RTTxq8GrQN2gCROPaLphhvtDemWEhuPfvi7f2BvKJ8YNxaqRRRoiVW2JeVdAuqkuPyu9Fho67qysqoHKeqhh7/9nUxzkQUQNEq2VifamumEe929vb3RMlNUF1SOqr709ZTBBhLTVHgfANMNzPKjBD9WJMg+qGWorstErGQE13N9YeKe2BmJ7ALTISnl9yLh8z7+oVxfSq9d4KJdL3XSgGFtc6BOBsdPWTeURybv1F1BEAG7dcCrnUBMR5tfyJ+/VstmGjHm4M4ndEruy/gS6Jl1GRAzW7wAksUCYyI5F8eU1fUa3ZZSrK9IQr2HKc/1PmRKJ6KIdQFJmpeuFy1VRIU0CzIjDGbzNVJor60GdKDBkgxrX9fyjbe2Hr95Q7to1tYo9q86zJHJurTYHCtOsEUlLfZe737PZ/9Geogc3OqthjfAJdlR+DjsAhaM9g9YhQyvt+5M/YRNe2xRJqbuIQUzhamuajJni2oWuG6wh6hk2CwNfFEb7qjn7x08YEt8p7hLPymYCXHINDdhzjqCxUPhPwdcHK4zCmNMMlC9yJ/VzqTsqOBQQVcIkX1WFuUOZsOYUJDIeLJDa4Ve1FHXF5OOMow7d+55B/543G5aqPF4bN7e0sZSkUZ8r7bAy/Kp9vhyB6S8H1/0kcX2DuC5GsfhIYw8CSYpuEQOEjBbKMK+Zvp+yEfwd1HtZK9KZNdb4zjOkBI1kyCrVhEk438akLmE+2CL3x+vm9KBuZq4tCWbrkF5777EOa7nPu8oImpXh4Ub2CkbUaIoMFIQKUbqVpohv6pt1DwIfTcPtPu5wiU7UbuDc3XaXcwoRpc4dJk6X/IUU7QCjKC601wMrCDygwlunkcizxJw4+luGoJNvD52gpQxGJagn8fxJBTP4K4tK6+ix2QVagn1dMqjdAddE8wodlLzjBlKJbdEqksGtaXkELzb9Nz1eXJmNCqBIXM1QJMchtCaGmhwulUdHNGGS9yCrYgZS1jG0wKQalKdJKXLaShSOur5lY80CP28NhZijiokKS2L8WRBFormvoztkXmerUN6Y4PVWCdidlMBGsJH+mpYAG9vvE+DrLddP9KbQBcY3wD+StR6U+GdgdRS2G0YvV7y9S1XLn/ttWZCWNmoNm7J7yNBR9WNMrZoX54MKeDhTkYWefT795XhKyviyioBQTZFwfAlkktqE70MrOMTRqeTMZccTzRg2i6y94D+wujSdACmRKNgeJcOLLs9eLTwYy/H84IOErWDR3DSr4IUvC/ft1s4Cyzy2MZolS9mRlElPaTX8YUtQjFzA39ggU1g1ba6ockrgAKaBWMDd1h90N9FS7sOJ5WTCJWO3mPsUpUHohTMWm+3Px6NJTawINWH93kgEj+ugNN9P6aaOXbqgTMw2O9gIMOVgYD9vS6eb1TzT+l0Vn8J4RrOWz5K7YbntTa2pSrVWCvaqjMumJBMCgOPNQIZ2kSGA+sbmV786fuRmS8TM5sO89IRzPYGiWuV3nATfxqHIDcH1tN3Tp9xNhOgQ8/Buj+LcEHkDNIgxKABT2Zxcss+cGdrYNH5SNywEPKcUIc9++rbP6es4pVJxevrFqzYpbhiYS4mVJRj/4nDnskaJlyd/hRYLmISfvpD6txyGlR9a7CHWfmXrvxx0F4JQWc6wyq+pJxstYbPKmdYkLWv5sDz8yC8zXAIW3ScYmC9fv7lsVUU19rb764elKbFY88DVQlYsS3jvC2z1bHcTnEqF8DG7zjWEMBd+0bWXNUH8vUNTbUN14Cmp3IkVfysmXZvb820ixUU46YlOaHJoqhwftr+KI15dYfAcz9hh7LOwXJnL2ABkeDkGtf6uxVwV9cvhy6eJPGcZP+NSBjKHP4OZXZTkjehBU1R+UAme9MTbLMJG6g38Z0STtEB3qIfdfjW6Qa5yzxE23vVmLUkBkxEqsggcoh1K54rT3Tar4NJwV21E2crKEri/zWGAWyqJoDHDvHchxz4sawESHVt5GCb63ysTVrVek+6gdZ62jAtAethigmqj3AUKteEM1XsVyX8mdlFnVsIeJV6tEvpRsye5VhedhR7mB2BwdYrJhu0Wc5kJtJtgNvjMMYj75Mg4XhuZES/Rnj0ou083JWzugERKjldLtBjZULacuO6EGtL5/1vqYVMlfpEqMqqeIAcMTAPUGQrdMBNH0yFPOSfo0Y1Xu7eVaXXaYdVfMDu9vbbtyP441BmZwge3oLYpXCS6HwTggmW2F0H9O9hh/Wce7KUz0DWU1HonYksidPbje0VlkbDWenyvLfQdUaRhTWLrsr60vTpQOuKMr439FJDBMx6zwFUdJ37BSpKTKDzH4HZyv3bYeMxDBFfugUSZAqQIZWGK8oQWSpJCJgXjQ5dy2ETqdwkops7UfN/RYNCO4tSnG7Zg2QvPVwse1fk0PERuBPShKPn7PjlDctH4exsSWyrD68Ukhu9yGL4J0Ar9DmIW0sHLRmUdOnLb7HI9CcqCnDY2wPh0Dvssm63K0tbAwIFZZfhseulvPBNRQV1ROXOZgCSKmGAnL5r9PPAoHmsp+hz9eGYz+tQpnTh+7JAt8PkV1367FB9maajRxNSdeP16wQDq9k2QwvP6SDl4qdj8F/8Ngxmecslw0NiahmHuHynw/3T9m0XTK24zmCz55ftmweqCo3UpI6qoVuDqHLiPkvPFAT1guq1g3OIGDFVASCmr9bys7UMfYxGSBWGGQAUasGik+CWR/L0jnwcffo+C/y1Ev4zXJCipNA6If7vMKEvkWmDVnuT4jB4kG0dRL0+smTt7cT0r+RXKux4Tr5t2JEltYnMb+WtGeV51hKTLrGrmr46OfhcelIE9TWev/JAPmMRlL4hYLFaeCL3ytkVfeFLsCz2kYl5yipftti6sQhRjaHpM1qFqVx+SktNUn0R49M/R3RQjCDcxIu6aNO6labVQtMyZnh4fDXEUaYMCVm8sogXNVc2towI1UREaPUHH2Be4WRNFyTk1h+oNr97RcepZY5Ce9i/f7oSLhYtvuzf5Cg8lZ/JSm0ssEiR7XQwJF/rtLSR3wlos5LJK/G0V0Eqw47VKvxLPbtLmD2ms+fBB1p/ibd0tgZvE3BcCMBzNFqSK+1evwUPhOVRLD9kwLHAbxiravj69P4tOEVW8aYwk/LNDzESSSWOZRaJuFCMvRLoFI/PysE+C0KZRed6cR5lYCHjl4tgBUqi//cvT4ycq02QrvPB12H8FiKYSo1VjNt9OkvWYVXdiodxyc7tfaaKPaGP1dk//PV/7XWxIBV9j+gG/pbF4yqj6+HhyLsHS+Pr6fHt75VxAnp9TR9UlK4KXn+FTv5b7WNP94HfqrsNGtR4HpO5B7oC7aRNHKVN1k8W7YTGX+ezEc5gb/+gwMAbWUeAqukIj8o2D24INhWFQddqpKIQwTqmVbX9qgjGT/2BF4k1IpXfdFhAfQZNyAOSBXwxUL0Gvqz6V4G+07sPcOmPho+HaLUaDVCapwGbBSgtsgS1tv3xH9s/zVIUZQfrhuwBjuceSpJel8Z3cNAtzFjcqdIv3toZq9LX67KY2UbqUxU/rGDwQH3ZEYalHlua6kuyUn4qk2Vvbmc8fRXDmtlmyvKtTKaRq4sk3SBg6TZGW0HHWRTZtUot98xIlV7d08bxvnp86UbIP8o8uhH6kro9oSwVrB8LyL/xdZkebgYWb3rjthpvHVMvj/5XImG5/JwOnpLbZfLI+wZmQ1KObMVcPmNrU6SeiKjE042bmvfMXUw89dSktLFgQJ7WRrniuxzb28omxWIRgBf8ppMqj7F5oFFWMeyojTojKHCFlf+prJwZclFfSd3UjsJdXGU83WtQNdVNW9wBBwM+3WDztldUVsSdmrIW42pxMXOxXKRE7Iu3r76yl6q7H7/PP33POFXlwHocRrX1DSyxPbOUoBxTEVJcN6oi1LHhoBqLNG40wqbi9GqkRW7r6pHSqbHmgbZXL+zzJM7n5eEFaTSv6kYdTavRflMdrccymUhW4jfKv4DAxOpXt4h+WBwR7LCnRhUtXa5G1QsDVTzHAvw8dW4wyW8wCyjRdhzUQ1ukSYoSaCTxsfBHIfyLCnFbn2cIvMXQaFHUAWWlPGOxxhIOLpcF73LFIqsji42gaFdn6Aroe83QjcJZTacqHpePi3hKuw/LD4v0B4xBIQ0AT/7ys84L9MzzAiumL88hFvlE61a6oGp6Zx3vYIbNe9J5VGDCVsW+wKGn79vKVKqyzqm1oiLp2uIG5qqYtsdNZiqOvagzAYOU0ZJy3ccWlpRiCxj8dUXPttu3A6xchhrk324OWWWuLytxq4EtodWSbaiI7AnobJ7gpxJtMD+kGMUEhy+KuqcsBexJW3SjBWi0cVYLxYr9qz9xU0ifKWdFfcFNbGHwvZLInfkbIGVN9FJ/FoaifRwsfPbxj9VA48c/bWSabygLz+mD63LMNAZ72bItD9LcX+Omnrv7d7sNgJ7rGN96ME3bkCoJFL+Kc/ItmJgyC/KmvdKZ3nokBVkvQFpIi1m8ymaHJ+uCYzrtttnafRePlOuGfnBl+b08cf0gqT82UoHH+OGp8MqV+2MyMbFWPxzm38HiXch8OhCC7apyUqfs69a4LZMZXckOHG/Ko4mwzU47TP87bP5YAMjH0/YKQKq2WNRnERY8U+c6S5BGPfhqmXSUJiTuUbR0yPrsKFOxI42zjiLVjiS0TilZOspYp5obqwSFwn1HSa9OYQSUeLNhzeW3kezyqzqN8IY/06d2yvDXT/q9nWoc5Kf+6I55zkN9dEd/Xsdc8NO1mASuKddbrr5eVJz4KeYQR7CGII+9qzK6v9Vo2pQNKa8dgzlYRx/MNww2bG/T57kk+JqKc9Lcw8oRtlnGXY1uqMjQHN1txsVu0TPxhOq35I8f01nZm5JuitJN8dhRX3GBewZ7SpsJq62S1UT2RFUmjRQsxf1Apl6HvR/QW4P4rG+Wl1WgPJDEw4InS2t0RahtWNocnULRVj61cGnMSZfONLso2L1TlTENUqSAv1KWNKyCUcvGGInESJ/ZRp23TlnPp60xPpzgctMmhTEruOnIukd2WSpMv1KSRfnCyOia6hsC5JGeHkXOOjrMdUqPCJ8UW1pByUbpHAN2URxID4ZCOu2KIgMB4lJtq/bSF5pWVASt6a5mlDWVdDT6ncWyoJbuvQHBSwREb4H1lwOBijEHu9G9SYag+0K6Xx8SVWNGUFs/vnQ5fUcs2XLfPH39q6evq99sHIPicBF3Nv4pi348DvDb0Vh5mgxZPFPlCawnjMAwzpDRJwVGeUqbcwxcaTyzEwajZOkbg2nsnYnyO01nshxfEKHHhm7NhRiZbhaavHqs9Lmms4bKej5+nwBLhFW/UUD+eL2WJAEFWotTUYsMri4JUxgkughaVh+VleVRJPC0XTk4LC3VPLDMUd8XzPQYEBVzREVCtg5iqsMIX1+wg267li8icejIf2RMLd1g2qkzCtBRs7oO/Q+NpwY/TH9BsAk537yhMqTLsCulGDUEGL8irZDnkTd15TLT9Ab37t/VE6OJDpbor+AgeFbmApZ8JYHalLyc0Id3BsbU1G3qS+JT9q5KG2Eyvlh1WBd3fkFruPLTOS6mCxWvxRculVlXN/SR7l7TBiHtIwvfpcTnwbBeOs3c+VcCQ1PVULMEfRMMWX9ZFBDPAiXn/jiEyRTiUFziHpm1q5Lad40m1poCh/Ca+U2yaqkrEFETQafId97T32/wOMTlmoPO1jTL5ml/d3cCJJuDHo9nxkjMQe2CLSh4KtJdzMBMs110blHC7q4O8lnG+zuAqPxyh8/8uwcoQSmdrFaNEubqTUHm4yw7rBvfOzysSFe4XSVVkEM28jtIZp8+5ZinmLPDsaiNpt9etXKvbl6tmoh3lr6jlydhysfCPmr/32QAk+YZz7PpwMZZdYpxNgSjsJk7AzLAz4g15cptzis/ij2kYqQojPSM9w5nI6vONKg9KhwDiIaf9FHTJbko+aTKQWbtY69a8e2EKmkOiYBKeW/t7ETxDmApltUk5S3odpO40piYBHikt3eP1q3XX+BSXW+yLywrsg3MEb48eYq2JniGiXn/zdsngFSjspvxlUssfFd+iqXUM/RB5epHWjL2kN3t1qoXy2Ip2JMqEecgGYTV2ojkG+B3ewQYnRiBs5NCPAz5zofuztHO6RffObA85Wp8hyIDtxGxC+MTDIbiny0rIrngM2dCuxINwSCwSXj5LbdVH0fT1vWQPmyK6tlF9WzM83TNt9OqRKlo5VThaZM6ZvLTb9bjAhm1amUofAsrA6e89IEmBeHbVH8xW0SYewi22Qjs7Nj4HDRlMyYyFy12HHMDhKRM3VZqFkhLQolkjRQEzaJmteRoJP3b69KSHGhgNCKUvA1LtCQ3lFqA1lLylwPf+j8syg9U", "memoria.py": "eNqtV81uI0UQvvspSo2izHidcZYVFyODlv0BxK5WYpcDCpbVnmk7Hc9MT7p7vHasPAJIXLkgjpw5IHFB2rwQj8DX3TMeO4lAoLWUuKenuuqrr366zBh7KQqlJadM5FRptRGpVSNa1FxnnHJFl7UgvKoNdqQiXulazDgpSpXWciFo46Rqw3sV15xKRVpUwkpNQmulhUl6vT494dCGhzq3PFNU1SITeE7lDIJ//fzj90P8+4FEOSIrKwU0FkAgOqC50gUfkBVrW8PApkckDFDlwqohL2XBU6lKilL8S1UhSuuBqspt8zxOYP4FNx6vSN2mMNAf7ESV0AZiQVE+TPlMXPGhml1Ae0xGNEw41SXl3MB6ITKZca8j54RTOU/os1yUmdBOhHIBVgZkJOW5WHCqSyoU+FVeSXMcikAYGMB64PhzLHtMtxgXuVzgK1pBFuwWzhWhU16qnWvCgCpprEzxELnzhbJKDyst+EVtrCDDc2yKC6W9U0WNIxpeidKZqngp8qT3uvXWbX+u1AJnnmq5Es4VZwNqSxeUj91GidgIxwmlXCPiLoWcHedQ0mOM9WRRKW3pAgS3a2XalT0HuEyWi92GLESvN9eqoISaPfA1l4teb/ri1ZOvaNwdSl6odBnFvefPHr95jRcR42V6rtiAWM71wi9m3Aj3XUmhS278ktuwmGl+pfwqRSblisW9Xi8Tc1Bhz6N4hOgQZd73MbEhcFgk1tDvDF9uPCvMC8k5nErcsUSaTOrIyzQa3EcL5G25E7pQsgwyMM6rR9kUJYXckVf8QiSOK2Bx53Cg4EskizZR4CF59c2b6dMvvwbva0R7qpbjN6jGIH6fmdvH2D2mgtu54lnrttWbDv1bac9RS6KMAjMxIcXm3fs9005h4hXNAySxTkVl6Zn/Qt3d4WTLVshTw0Z0NgE6l/3h4bqBZfhKREWDq0IoAogAs/Ab9IBYgnWIRgcXW1D5lt0G7FFmdVFFBXrLgCTKtrTjhy6ZTa3FlJtUyvFznhuxCwRaWs5TEZRWLWlzweGGQHhQ9Q3IxrOzOYiw/kWyEDZaDugU3KGX0RImyeftpNEDEkSUOShz1BRCtKhRZgNSONV1tDFjjQ3U1uehTtFZHIN7HRQ1iM1dp01cHbozHFw5/R4NC13B5f/2Ou6I82XWUVXgTEiMbuusCdkk4RVozqLtQSowy0a+khP0l7lbROzo25Oj4uQoo6MvRkcvWYywhHYI0TZdXa2WHNJZPDhU2NIB4Y4ZppZ4nimVR2rpFHY0Yb97OBs9Oj2d3NLoeyPEeOAiPELHeV1ual56gDBlhJ1W3PiMbETBWWXYbYRof3xPaI5EMV6Jy2fsRx3vLh/kHG3aXUNITaQDAhDeeelD1e/n05nHfc3iPfjXXWCbSmsy0gqenk/tphJRSEoHbhpuUNyLuzR80swA7dUVsi9Ed+QuMuQoooFL7EIUFS5APyPkyuACWdQSUcJ80KaoI+cgS2+x1bbbUlkve6ed+Jr9j9kcek6XzWyOiB0WdhfJAxb+MVS4jlx9jWjXAYIOr+zfS+SewOz56Zp+2yAtt2Za7Hr3LSdXeCyC+VC2AzTXAyJXxMvMr4oOpmF376+IPb75o3Tz3Tnf+K5jEnoqTFXf/OpnodSNeM2cA+4qhcZK/f67P19jJrj5jWYI96f9PrE7vLEN7Q2fJPXNL9TcVPhDDwuIc+kmtzF6K+v3t5i3olV8HZDQu9/J7wQnjr0Tx95ZiNw7+TUGMKz0+2ziDbjufOm6c1uMxPzw6RbN+OmWu+lzn6VLj2ztdaydjpVjeH3W9a8Jjcd0OdmdwOtLc3iRqiWUmLqIcA4dbtJpuzTxgaTnok3aOTuh7WWS8kpacH0lovh6RFu1vB56UnD42tNP0fbh6Sn1naHhkHbvjuKG4g9oNzzujYxuzPNE7FgKEp6ptms6jrSjffE/aGl49rm43m/LsXvrlwfENcjpkzF9+N4ofO4w+BF96y06Du+nqyVzt+OTsCURvwjUvW67Kmth7Xm6d3fFk7OTj0aTtkC9ps6/A8Tsu7Lfv/kpRwOBNRTPLBfut4ELIH4bYM61YoTUjg+P0wNfQEiY9dlxG4TjifMVGx0UbLEOvgcy2e9BMM/CkOnVxr2/AU00Vnc=", "estudio.html": "eNq1XFtv20iWfvevYJTuFplQ1M1ybMl0JnGcy8CZZOL09ACGkZTIksSYFzVJybIVAfML9mWB3X1bzMMusA8DLLAP87YPk38yv2B/wn6nqniTZHe6e7eBbpFVp06dOvdziu7De27kpNdTrk3SwD/aOaQfzWfh2K7xpHZ0OOHMPToMeMo0Z8LihKd2bZaOGvs1AIvhkAXcrs09fjWN4rSmOVGY8hBgV56bTmyXzz2HN8SL6YVe6jG/kTjM53abcKRe6vOjkySduV6kdZ8dNuXIzmGSXtNvP46idNloDMf9++1Ou9veHzQaUxZyH+/DttvZxbvvhbx/v8M6bncPr+ki7d/n+/yAc7wFM7wdsAOXMbwxx+nf7w17w1ELb9ElsIwO3L3haufBchgtGol344Xj/jCKXR43MLIinpjDyL1eBiwee2G/NZhwbzxJ++1W69vBkDmX4ziahW5/zmKdKDUGTuRHsXoHNcZgBLb027vThZZcJykPGjPPbLDp1OcNOWCe8XHEte9fmQkLk0bCY280iOY8HvnRVX/iuS4PVzv302i6nEYJ+BiFfTZMIn+W8gFGQcx0MfD5KJVPsaIQj66XTH123R/5fDEYs2l/H4PM98Zhw8POSd+BwHg8uGl4ocsX/d6AIBtXMUDpP6sdK+HjZQXNxqmFSIyB5Fu/TQeNfM/V5CwJKJtsxMz1Zomkbcpcl/jdnUrSOtOF3E4bztI0Ck1rmIbL0m5pDPZMWQySK1yGlPPdW2s7PSpthGetjV2kRLxwAkYD0yxOgGoaeYITVxMwpoFtHN4Po4IHiigrqpB0v+N0Rl22IfTKmj7Yx4Y+d5cR0Hrpdd/q9rJ9XT5iMz+lBdMl8bffxrPrFqKOuc9Sb87F8NE6U75eBFtodF3NCng4ywUcRiEfbCqZ1KiWVLb2Ns3/lTqwh+fAC6Wz6LcPaC5gi4aytr3WfFJYBJulkSDeiqY8XDvB0I+cS8zSoMZM+SvlUIUZqL3IkFO+SBvCLPpkRps2LABc7kQxE5yRfFLE75NeEcXVA9KZSnwSS3ItvUMFc+L7Ezpy5QhyaIsGZouSgPn+csM6wDeW9oUY4Un4YnqLHkWXuZjKOGiYKG5cZfJoAc0wAknBFp8kJ/rC1oRb6hGPyXpHURxIO4ZO8z/qDcwYJe9TcTTCFblezB2BHgTNgnCb98ocG2mMlOrB3vwKFLLQC5LlhhMkyVT93ODTLEm90XVDxTCFOUORKdCvtruSq8rUpLNmBz/potaIsrbRhVC3VZBiXA7cH41GQOWFo2iLCGU0QdASNr9HNG4qFWkEgiaXks6kuDugc2em27Z62CVJ2ZgPmTvmS8LXI2kpALLmkuQetYTkfD7mobu8zRWJDZWaPVpzFru9dWdRNkOVOTjOL/ZVubmvnb/iQ3Nm5Gex4N8015tXfG1x0g0Zy/HKiooCb7EDlaSAGK21yhB4y8zRFTzf4q26pRN4oZCgcqX34Tu2ScILkRPClVWo20LVdtPaVKeMZd3VjsPCOUvWffpvAu56TNMLZdnbxUGM5Y4mkqOMM6Rh+5nr2c8Ton3KL7SNBKNsegdVqXbFCmkihFTqktS67RuUI4k6Zwlfu8CnDeM1RdAyf6o26LYkdBpFfrKsurZcvRsLFQ21qqPKDtXboqvqVGsmWT5bu9eqHKqq6LRYKW0FvnrQskV29mCRxQl2xQmyVOigs9pZ7Rw2VdJ/mDixN001Kk3smhdQaRGwae1oqV6SWn9ZSycx57V+bZKm06TfbDpuaH1KXO5789gKedoMp0FTAP2mZbX3WlarOZx5vivHrCByZz7HipopUTXBryhMmj8PJV+wAGl80vyUBM3aaoVDCOJxiqYsn6h2wBuMWPNcuwZGo/LRNDHg+CxJ7Br0sSYmqZBKxDQApBw1l6WsMcdCvqhlC6KwdvQeajaL2WFTwh2twyNLBqOO3nr8hiU51FbUMqQTME+i5FaM2Py6dvQkdjzfv33fiQcYP70d4McZc7HXMX7iqELZYRNM2WAO+LV1wnUl03wivpZvo2Zh17Wjv//Ln7TT2Y3293/674KcEgrKl7ax26cCWBSltaw63X4cAlzAxL0oBqR60nS4R+MOlmMVfuB6HY5lT7LHW7dwYxYg6XRY7egZHr/8mZ5vhYZDCQH5ln40PUYJnrBYS6W6FGRJnt7JWYcFwXbG/s+//sO/acdf/oxww342dx271jJbtaPnVMbx7ecATHtfQJ2gEPNddivYgYA6ZW6keTc/WnewHcCNEjQin3Ur1m7PbO/XjrrN3VtBWuZ+i8wh9oa3UzdCEQldZ2VD/WnGIzvfzveTBfk/iPNOnmc4pAAq+2U/mTuiwC7oG7MQPIFX5H709z/9+zpcESpqmnDTIPZrCsZKWtirZGGqDVOkopsp8JY0uRrFBqJpM4E4r/otraW1KTNTmPf3c8dRHFcErc1xisebozK2Fe6axkSMvcUj5X5cBOw11afxOEorPvwJIm4Dg2zT9oX9pXAPxzCSuASwgfQK5RHMPfK//GUchVv8t5DfJffJln6ccZ+n0XaYCciDaf/jX7VjNq0El8I5Zz+VGC2DKQ4s47P2QGOJ9v7lu5MTbRRHgVYXQbM+yOaXb+Khlx5DlDEYtSoDZWHYUZPNCijCcQnLi9P3z0+hwEiWt6HwxVTSLMCqy5+9e3L85ivWl+CqCN5FUXASzr04CmFq6VYkvJhPmmsLJLYdHDVJtW/sxD5yI2cmZiCn+PoMsnLSKNYTY6Cgnr86PTmzl/Dm/fO6tNYPV3xojf1h3cwGSEdhEtVBGQC8m2ziwhQZAtBMRYqgBlUuQMOUDKhRCv0YYjL2q8GJRzQwDHxgCPhqVIT3gjjxKqdW2SF+9+r4xF6uk9+vvzh9qiGa8zjSMKjpCHZOpD3Ufvv25IWx/XhykRrRdCwztReRG6Wm9l7keZ+S0kr4JGs0XGDV86d/pGa1xjOL0PTvQySjpvZ9GHPmm9prD6ldhMU7mlpuDX04Dyx+Sr8ccV6OGLczWZKXeKWNCmCkGBTVIysafgLkm6e/BVQqontU2vYD1C3mCZxr17WS1Afo2ftTTY1++a9Q6z5DRRSUyPAjt/2h11IknL55lrG11/q2AtX50O6VoILZdQbZ7n0raCgpR78uk0kN2q9NZlAP4Cq0pF9/zaYMYVUTY5grqwuIhqPXqPvDiISq2tBaDGg0oOFYCCGOOJpOvIu5i9e4dMBCq7KVHIGSMkpwPsEZgqknkh+k29h2/OUvDHqkciBtEsUhcnNGCOeey6MPyHS8kMVWMN0Fxj/QmFQqQUUONobldpH3l8Hw/rf/FBAQ54cp5VvMmoZjALwKEDRD4oiY1ORkfZXbfCz0iMd2yK+kv7R+4MMXp+/UuL5kId2ZeCzpp/GMmyRyHs85LOMKkfLpbDTisZhawUVk6FDgpm+9BfffUatSf83SCbUedHkdU8yYHcMYlBedIa7qXhjy+Ad1ZYPHl8IjlNFHs3Q6g09G4D6jLrktST979+JpMVggRgDhUI0pCFaQT45Pzp57fuA574u5EiUioGPY4qHomtt0wsFO7h2pprKwCguOJ6jp9HypCwfrcwLKXWaCKpyXOHxG77oxEONWkZGUQMQp9NZCXjnlmKZBzIMS2NvX705evwCWmJGXzoig9hqBw/WjSLPFKotCg9yZ1q8FAt0wUUzuGpbUUJ5tCLUsb4dgNKVO6JwfswCb6shSC2k1S9IidG2z02rltDupL1BVwqmODcytvAO4Yv4zFLgkOSECGqY2wTvkLCnfNnY25ZBX2+pkG49jr8zaF3h9yf0plLvdMneRji86vNuWD53ObsfJJIMAqtPi/AgjP4rKpvKaJxO9xB4UPPwFjwKextd6p4XjG2ZJ7EqpqEZjvl6+iFkZoj0exRblYpTGWgu7Iczm7atmJ5tDjQbWSzzy6AWhAiSndMIDr0ToS7wm0wmYfErCgWKNxD84chf/7O6a1l751LS8UN9ZWEL1LOuFM38NVwcYZqHlsCStkIgxaU9WwKZk4GToYM/uvkn/Ke8L2Hzb2Avu3nZ/nzFs2y4jwCIg2EGYgzOGqGa+bwZwN+qRCRSJvVwhoYicy7LF0TvMAPl8afRptOhi8CX0ad+kFFZqjsRGOTuhYnPmSd0+w8HyA8jsTYwXCaCe2aZLqYWYLGV3mBXjxKFn3EGoid9CCfT6L20I+d4waQqUzbGfjpp1YyDJEjuUNhYwxLrRLBRM0kRXQ7+kzqYmST61l6op0T8vpG6ed8xds3thFhKBNpltq2WS/C/MrD+hFnVc1+xaWNYzoQC0bDhy97HM6pnWAS2Vy6jMyJsUci1r7cGoAHHe2MXGYvFe61GL9gRGGJL4V67PexYlYrvY47xr9sxGm9aORr3eo0cmydYC0e2eOWJ+grWig1FaZ+2a5y0s68lV0nSwbk/sKxetzi8vBiBa2AC5b2LxS77QT89bF9I0qK0eoky9tk/P2xdiKCtbhU1YlnV63rkwrGDmp97Uvz5zmM9i/SWCpBesY+1eyOEy1l1BAo1WEDc6D16a9G+j+wDIyLwry3oXyoZKxQKGH108liGkT/q+CfKqhGJP7OyN9Hu0zlhu2eOhtUdt1lzDhrA13aBbB2s2deEX4fBib/FDFCOgkhDpCmQBr3HJT4Jpep3bznDDQmlvwoNCdY4gxfXIPlqCmMjyEvLSRvZydklBypVjsLNLkRKHigAgisDmAMkFf0ohmVIcInIwxPD0GiiGxbBBGYB/LanexVxQ0G+sRJnMoRnYZqyCwk/gzsG+ehPizywEM/WhsVpRLoXoDc5GixKfR7AErrMbG7bBfbvdETYtua846tiEaczTY3FhUApqfxAFIFhsmEkGJLK0bSCDl7bM9dhCT6xrEzmAFI10eeXJhUkAiXVjPGhbB00xk7KQEgJrFM0fZMEP2a2BCJgjYTa7ySepQ8dt7pcHBChbMwDHWjwUMMjmdWZIeCdKdG48cE3Hui5m1cjNwxxmHV4eCUkHvBO4IaVHd5rYFWl9bLtNJD7idURvD3ryRerY2zj6JEOZlKkuc51cAcsWQvVZAhOheJbaLXNutwZ3qfl3392LrBkS9GfwnBb1ZWBeMgmyC/UazB/aY4ul2H0IZUwKVjnQu3SQ0rRomD1Wv3Ki2e3fsazZJQXM5fQVygLYb/S6uIqqG5ZMHt+/PrU/frMU/BZJsZ4iKY1OI/pm6wxbh2PdWGl4+PLncDzzUWv97a/aN8v5FqD5l/+IEQB4cjiMj75ZQucA9BzZgKt3MP3lnzUavN42eFMZDD4+JGvWiVJL1J3J449y39LQSpakycd+vQ7zZ8l16Gi5KPkCFViij4wlJKCEEtvsinnIK3nqTPSRuYR4JpHbr788efKsXpgzCp7LlcMICAJVgyLqrFY76xslk+hKp3ucUuR2WOgmtmjdnNPUxYA0auT5XGQyAxTFukpxtWgkwQ3olchsrAkjuo2lgB8NhjFnlyvl7GnMkBRl8qQEA/IUTVpL9WjtOt3M1gv1GKujq2yEfp7QOXRCCJsWas0hhZT5Rgkr1SfH6hu/j5tdY62iOu1W6wEXuLnbzJCtvv0o9RQbwJaMpQxrKJGiORcjK8wUud4aQGlisJYQrmQQgvGIFYO7A1K0kSlH2zL8CIXbDFlXcIwdUNgIsQ9EcAE6EqZt23Vqj9U/f85eJ17dEJFClBprFcsZ3KyLvKqoRGQHHInYwdBx9kwwbzwJeZL0rV7PpM8sffnW6knOVbZW/bo6neirdwRPIElR5arGQmnTRyvEMrlTkdmTYAbrYSsIhZcJvNC6FuzOXdJ1ww5CVTcVg3CfVNBVR4C0KBEGRYkwILUT3faKd6rXlfLAF1KDRoBbPg/HKaSqMOXHf5KBvKaJ/BSZGYjrBfu8/o6DJOpmvUTFy2I8vPZi8fskgap79PQ7PhYjZ1FMXRhqYj33qXWPhxcK+h0fqpFj2VCipyhGcU0LmT9zmXqSUE9h33I7FoqO2dso4R/e1y8kkZUjJlGc6jozh4Z9pAvCZXB4Az9h0Xe4xsODA+Pbg4PG2uywMqvOX7gch1xOeSdjqYRw7oiVF7bgquX43vSJmKFoK7BkfBzaeWvGgXtKueoj6HV5f1CnLKvsOyRmGBy8k8P1+gecXRNQUYh9UBrqOCa5Ll2CGhV1KLd+hirXE8AZ5ZlELx5nT/3yEVESKKzyaCr6U9NEYCP3DLd7BgejLP42Z1JOaLMVqh1HBJOLAr3Z15/2PQVTnhT3QacIT5mPqwuGkfnx9AckEToBUzZRAaYLEuaFiQTPw7m84doIAOvuonBW4hag/rguvrCp9+t0nVcf4HhFZ+lwb7dlVJAXZJBzqEPX65sUFEa7vvvj+uEw/9jgiPKDY+YyTThCzdZmoYzlFs08i6DHYIUW8BtsG5qazzTRPqZBN4LBWvW+UIH1I9EmsmON+ITYwONppG5Oj+oPBdSpIFY3+tKvyFwdVbqoviH0U9n7VJng4PYAK9kGFNuvcJ74PpaK7zu0zCaQaMcnDFnF0D4alliaRuOxL9XAHFp0kwxFsOY4mcgrqjlq5RQqsUk4lz2j12xKzvoXpayB/SSO2TXgxK9ehBfjcfEMS+oXb8o1B999Fwj7AnaiRaQwcsQwxAAVBXLArN+vP8yK67EorrMU0lBRSGVc56jPafEFNbJ0/Tw0nQs4iY+H4nrby66lS1fM3yydFV3Vekg/w5W8ufxoWJ8i1BkiSVSNKooSMvyU+EruJMyYgbJHeZbwgmzjHsuyLpnGEIbvvlMP92ybGeoZJYjL38xS3eoaA2ZR+z7VDTH6KqRBS2wkhCTIYD+lQ+WPvL5Wh8qu17bDDb8LVpeqHuV1olCU7fcoaKrDSmZcJfb5xS1aVVWlK1Qps2SiRwYkiZeM2Cv76MqSn/FnTu/KuLWLcC9T1s+f1zcoUl+heBstBv1rdLh/XjxfFBwN7KNA7CKcgg12qIJglW+X68fVelu6KPfM6sxTlnjOZu63P9wfjkZmvp3Mykp/7iAHSk1r0+XTdPJD7KW8L4sRI4/KV1U2yVT2ypKd/jci6ekgtSXffUVWtsp7tsLf2XXVaKyX7lw2VTH7W4R1LcxCOJURHH4ymqLonrKxiL1FB9YeKhVQyYIaj2x3e5RDyJdB5m6SStQs7KPFtvAqMBnCjiPDXQ9mcppcT74Rxk/meCAguu7R6+KEdZOylP8jcow7eX0uv+66uI3XRMgyl18RNfz1ULb6mn2cn9inegskMyQKjXGU3pnQeJSNZqQ5FKrp+yjULj+7G6ZAU3tz0rEWpmxcLlCHvHxgtR5RQylL/Nd7UynAU+sa/948BHCvJxtC5eZSutYjKtyApOKcmciSSyezkBR4yH5NcIOC1e9mwZAqVtUKNPlPieHOTKEsC9FuKCUJpEaVJKwMfEsKl8UKlcUVgrxL6Peq71vFXw5CVXBjJfdx0nhtnywD2yklvVU6pJcI7XtfkRYPtsKU6SK3Xop4ii6Vmt+x8UbyvmXjDZj1jXfyMFLuZRRXlqo9ru5LRfVaAOYxyxKR4D18tjLFbTD0Ob/8o8xO+cav1E6R5nFn0+RXhtNlYJUimoxKgaVimi0uYkUwUo37r238DP6/ic4obA+qB1Bhl2jO9GayaTj5/bp80MWxTPihzImxW6tnVqekcYKcwN5ySw91ohD//btTve7RpyZN+uyEVrjRVUhlil135Id84oMUTAiydNLyzZBGXxPdcBnTluQmmfjawN7+hcHdLfWv/ahk/auVvF9zGkXI8Quzc1NbXBlTWHjG/ZTpqnZHqVEu4d1NX30b92nvHV20b8VGUDvqDYtrZdkd1cv94XrsjT+IfvOnRFTc4he8zDvDRUBAdZpfSL8ZEnOsOfNR0soeNoudiTePks+flyppW+sAb/l6buOTKfPXff5nlr/sKn/GVf1s68LArvILh22EqsNd8utEp6/6jDVwyUAoSOAl8DgIbVTF4VcWcap3Li1WwuaNekN2vyUuxaZfGjGL/otswpcC5oWVIE/XQUOl2166Sglssmz1OXX967iw0bz/GZY+KhswvRR9FEJ+Prp4+FE7FH/libp2lOUaFg49hdEY9DcoYvIj3FW5U8aEa12jXZBJ9N5TyH8dreVKc7R1f4n998I+4LrOOFnDW4awn+iwcKYSMxoVvKbCZG7/ngxfr8/rxufP9ZTuMohkdaVymwzndlkyAhiK4YWufmkfybWXt63FNqKrr2K02l+mEpS4tuvG1txClhAq31XfgaUZ+TxlU+IeXa6kGaOHQr+KD+w3mqU8xcBau0l261a5iclrJ2oNGZQ85X+GdNgUf3902JT/i4f/BaU93Xk="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25' sentencepiece protobuf")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and not os.path.exists("/content/TripoSG"):
    sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    r = subprocess.run(f"{sys.executable} -m pip install -q diso", shell=True,
                       capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else "diso no compilo (TripoSG desactivado)")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

# ---- reloj de pasos: duracion de cada uno y fin estimado ----
_T0 = time.time()
_RES = []
_EST = {"Hunyuan3D-2": 90, "Hunyuan3D-2mv": 45, "TripoSG": 60, "Texto a imagen (FLUX)": 150,
        "Precargar pesos en RAM": 45, "Cargar modelos en GPU": 100, "TripoSG en GPU": 25}
_PEND = [k for k in _EST]
def paso(nombre, fn, activo=True):
    if nombre in _PEND:
        _PEND.remove(nombre)
    if not activo:
        _RES.append((nombre, 0, "omitido"))
        return None
    resto = sum(_EST[k] for k in _PEND)
    print(f"[{(time.time()-_T0)/60:4.1f} min] ▶ {nombre}… (despues faltan ~{resto/60:.0f} min)", flush=True)
    t = time.time()
    try:
        r = fn()
        _RES.append((nombre, time.time() - t, "ok"))
        print(f"           ✓ {nombre} en {time.time()-t:.0f} s", flush=True)
        return r
    except Exception as e:
        _RES.append((nombre, time.time() - t, f"fallo: {str(e)[:120]}"))
        print(f"           ⚠ {nombre}: {e}", flush=True)
        return None

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))
print(f"Inicio. Primera vez: unos {sum(_EST.values())/60:.0f} min (despues de reiniciar, mucho menos: ya esta descargado).")

pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
paso("Hunyuan3D-2", lambda: snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16))
paso("Hunyuan3D-2mv", lambda: snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
     + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16), CARGAR_MULTIVISTA)
def _tripo():
    snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
    snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    return True
CARGAR_TRIPOSG = bool(paso("TripoSG", _tripo, CARGAR_TRIPOSG and os.path.exists("/content/TripoSG")))
paso("Texto a imagen (FLUX)", lambda: snapshot_download(
     "black-forest-labs/FLUX.1-schnell", max_workers=16,
     ignore_patterns=["flux1-schnell.safetensors", "ae.safetensors", "*.md"]), CARGAR_TEXTO_A_3D)

eng = get_engine()
paso("Precargar pesos en RAM", eng.preload_weights, P["preload_weights"])
paso("Cargar modelos en GPU", lambda: eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA,
     want_turbo=True, want_quality=MODO_CALIDAD_TAMBIEN))
paso("TripoSG en GPU", eng.load_triposg, CARGAR_TRIPOSG)

print("\n======== RESUMEN (mandame esto para revisar) ========")
for n, sec, st in _RES:
    print(f"{n:28s} {sec:6.0f} s   {'✅' if st == 'ok' else ('—' if st == 'omitido' else '⚠️')} {st}")
print(f"{'TOTAL':28s} {time.time()-_T0:6.0f} s")
print(f"GPU {P['gpu']} · VRAM usada {torch.cuda.memory_allocated()/2**30:.1f} GB · "
      f"TripoSG {'si' if getattr(eng, 'tripo', None) is not None else 'no'} · "
      f"texto a 3D {'si' if CARGAR_TEXTO_A_3D else 'no'}")
print("Todo listo. Sigue con la celda 3b (prueba de 1 min) o con la 4/5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba corta SIN página web (~1-2 min): confirma que todo funciona y mide tiempos
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_last = [None]
def _p(f, m):
    if m != _last[0]:
        _last[0] = m
        print(f"[{time.time()-_t:5.0f} s] {int(f*100):3d}%  {m}", flush=True)
_r = get_engine().run(_img, steps=5, octree=384, faces=60000, tex=2048, rig="auto",
                      tex_rapida=True, progress=_p)
_i = _r["info"]
print(f"\n======== RESULTADO (mandame esto para revisar) ========")
print(f"Listo en {time.time()-_t:.0f} s  ->  {_r['zip']}")
print(f"Tiempos: forma {_i.get('segundos_forma')} s · textura {_i.get('segundos_textura')} s · "
      f"Blender {_i.get('segundos_blender')} s")
print(f"Detectado: {(_i.get('clasificacion') or {}).get('tipo', _i.get('rig'))} · huesos {_i.get('huesos')} · "
      f"pesos por {_i.get('pesos')} · malla IA {_i.get('caras_alta')} -> {_i.get('caras')} caras · "
      f"cuadros {_i.get('quads_caras')}")
print("Archivos:", ", ".join(sorted((_i.get('archivos') or {}).values())))
print("Avisos:", _i.get("avisos") or "ninguno")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()